# Scrapping the DSE Winter Schools
**Note:** For reproducibility:



1.   First make a copy of the notebook and work on the copy
2.   Check these things: paths name, file names, APIs IDs, and customize them according to your environment
3.   After doing a major data operation, you can push your final version here: https://github.com/GaelKBertrand/scholar_info_append






### Mounting Google Drive as a working director

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Installing and loading all dependencies and libraries needed

In [3]:
#Install necessary dependencies and libraries
!pip install waybackpy
!pip install googlesearch-python
!pip install google-api-python-client
!pip install beautifulsoup4
!pip install fake_useragent #necessary for google search rapid automation
!pip install selenium
!apt-get update
!apt install -y chromium-chromedriver
!apt-get install -y wget
!pip install requests-html

import requests
from bs4 import BeautifulSoup
import pandas as pd
from time import sleep
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import logging


from waybackpy import WaybackMachineSaveAPI
from googlesearch import search
from googleapiclient.discovery import build
import time
import requests
from fake_useragent import UserAgent
from bs4 import BeautifulSoup
import logging

#from requests_html import HTMLSession
#from requests_html import AsyncHTMLSession
import pandas as pd
import logging
import asyncio
import nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.7/481.7 kB 25.8 MB/s eta 0:00:00
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [2,741 kB]
Get:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu ja

# Scrapping ISI: 3rd attempt: PDF MINER.SIX

## Extracting text

In [4]:
!pip install pdfminer.six
from pdfminer.high_level import extract_text

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 30.1 MB/s eta 0:00:00


In [14]:
# Specify the path to your PDF in Google Drive
pdf_path = '/content/drive/My Drive/DSE/dse2021_removed.pdf'

In [13]:
text = extract_text(pdf_path)
print(text)


14th December (Day One)

Introductory Remarks

07:45-08:00, December 14

•

Session link/room

Chair: Aditya Bhattacharjea, Delhi School of Economics

Invited talks

08:00-09:30, December 14

•

Session Link/room

• Health and Development

Pascaline Dupas , Stanford University

Chair: J.V.Meenakshi, Delhi School of Economics

18:30-20:00, December 14

•

Session Link/room

• Global Value Chains: The Economics of Spiders and Snakes - I

Pol Antras , Harvard University

Chair: Uday Bhanu Sinha, Delhi School of Economics

20:30-22:00, December 14

•

Session link/room

• Global Value Chains: The Economics of Spiders and Snakes - II

Pol Antras , Harvard University

Chair: Vanlalruata Fanai (EXIM Bank)

Back to page 1

2

Contributed talks

Parallel Session 1

•

09:45-11:15, December 14

Session 1(Room 1)

• Topic- Choice: Theory

09:45-11:15, December 14

• Room/link

• Independence and Lexicographic Expected Utility

Siddharth Chatterjee (Indian Statistical Institute)

• Choice by Reje

In [15]:

# Specify the path where you want to save the .txt file in your Google Drive
output_path = '/content/dse2020_removed.txt'

# Save the extracted text as a .txt file
with open(output_path, 'w') as file:
    file.write(text)

print(f"Text saved to {output_path}")


Text saved to /content/dse2020_removed.txt


In [22]:
import re
import pandas as pd

# Load the file
file_path = "/content/dse2020_removed.txt"
with open(file_path, 'r', encoding='utf-8') as file:
    lines = file.readlines()

# Initialize storage for results
data = []

# Regular expressions for parsing
session_regex = r"(Session \d+\(Room \d+\)|Session Link/Room|session|Session)"
chair_regex = r"Chair:\s*(.*?)(?:\((.*?)\))?"
author_regex = r"(.*?)(?:\((.*?)\))"
bullet_point = "•"

# Variables for context
current_session = None
current_topic = None
current_chair = None
current_chair_affiliation = None

# Parse the file line by line
for i, line in enumerate(lines):
    line = line.strip()
    if not line:
        continue  # Skip empty lines

    # Match session
    if re.match(session_regex, line, re.IGNORECASE):
        current_session = line
        current_topic = None  # Reset topic for new session
        current_chair = None
        current_chair_affiliation = None
        continue

    # Match session topic
    if current_session and not current_topic:
        if line.startswith(bullet_point):
            current_topic = line[1:].strip()
        continue

    # Skip time/room lines
    if "Room/link" in line or re.match(r"\d{1,2}:\d{2}", line):
        continue

    # Match chair
    if line.startswith("Chair:"):
        chair_match = re.match(chair_regex, line)
        if chair_match:
            current_chair = chair_match.group(1).strip()
            current_chair_affiliation = chair_match.group(2).strip() if chair_match.group(2) else None
        continue

    # Match paper title
    if line.startswith(bullet_point):
        paper_title = line[1:].strip()

        # Check the next line for authors
        if i + 1 < len(lines):
            next_line = lines[i + 1].strip()
            authors = []
            affiliations = []
            if not next_line.startswith(bullet_point):
                matches = re.findall(author_regex, next_line)
                for author, affiliation in matches:
                    authors.append(author.strip())
                    affiliations.append(affiliation.strip())

        # Append to data
        data.append({
            "Session": current_session,
            "Session Topic": current_topic,
            "Title": paper_title,
            "Authors": ", ".join(authors),
            "Affiliations": ", ".join(affiliations),
            "Chair Name": current_chair,
            "Chair Affiliation": current_chair_affiliation
        })

# Create a DataFrame
df = pd.DataFrame(data)

# Save the DataFrame
output_path = "/content/restructured_sessions_corrected.csv"
df.to_csv(output_path, index=False)

# Display the DataFrame
df.head(10)


,Session,Session Topic,Title,Authors,Affiliations,Chair Name,Chair Affiliation
0,None,None,,,,None,None
1,Session Link/room,Health and Development,,,,,None
2,Session Link/room,Global Value Chains: The Economics of Spiders ...,,,,,None
3,Session link/room,Global Value Chains: The Economics of Spiders ...,,,,,None
4,Session 1(Room 1),Topic- Choice: Theory,Independence and Lexicographic Expected Utility,,,None,None
5,Session 1(Room 1),Topic- Choice: Theory,Choice by Rejection,,,None,None
6,Session 1(Room 1),Topic- Choice: Theory,Anticipation and Temptation,,,None,None
7,Session 1(Room 2),Topic- Industrial Organization,What Drives Efficiency in RideSharing Markets?...,,,None,None
8,Session 1(Room 2),Topic- Industrial Organization,Production Networks and Learning-by-Networking,,,None,None
9,Session 1(Room 2),Topic- Industrial Organization,Building up Trust in a Dynamic Game: A study o...,,,None,None


In [23]:
print(df)

               Session                                      Session Topic  \
0                 None                                               None   
1    Session Link/room                             Health and Development   
2    Session Link/room  Global Value Chains: The Economics of Spiders ...   
3    Session link/room  Global Value Chains: The Economics of Spiders ...   
4    Session 1(Room 1)                              Topic- Choice: Theory   
..                 ...                                                ...   
177  Session 8(Room 6)                             Topic- Monetary Policy   
178  Session 8(Room 6)                             Topic- Monetary Policy   
179  Session 8(Room 6)                             Topic- Monetary Policy   
180  Session 8(Room 6)                             Topic- Monetary Policy   
181  Session 8(Room 6)                             Topic- Monetary Policy   

                                                 Title Authors Affiliations

In [24]:
import torch
from transformers import pipeline
import pandas as pd

# Load the Hugging Face question-answering model
nlp = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

# Load the text file content
file_path = "/content/dse2020_removed.txt"
with open(file_path, 'r', encoding='utf-8') as file:
    content = file.read()

# Example questions to extract information (we will update these based on the structure)
questions = [
    "What is the session?",
    "What is the session topic?",
    "What is the title of the paper?",
    "Who are the authors?",
    "What are the affiliations?",
    "Who is the chairperson?",
    "What is the chairperson's affiliation?"
]

# Function to apply NLP model to extract structured info from text
def extract_info_from_text(text, questions):
    extracted_data = {}

    for question in questions:
        # Applying the question-answering model
        answer = nlp(question=question, context=text)
        extracted_data[question] = answer['answer']

    return extracted_data

# Loop through the text and extract information
data = []

# Split the content by lines for structured parsing
lines = content.splitlines()

for i, line in enumerate(lines):
    line = line.strip()

    # Skip empty lines or lines without relevant content
    if not line:
        continue

    # Extract information from text using the NLP model
    extracted_info = extract_info_from_text(line, questions)

    # Collect extracted data into a structured form
    data.append({
        "Session": extracted_info["What is the session?"],
        "Session Topic": extracted_info["What is the session topic?"],
        "Title": extracted_info["What is the title of the paper?"],
        "Authors": extracted_info["Who are the authors?"],
        "Affiliations": extracted_info["What are the affiliations?"],
        "Chair Name": extracted_info["Who is the chairperson?"],
        "Chair Affiliation": extracted_info["What is the chairperson's affiliation?"]
    })

# Convert the data into a DataFrame
df = pd.DataFrame(data)

# Save to CSV or display
output_path = "/content/extracted_sessions.csv"
df.to_csv(output_path, index=False)

# Display the first few rows of the DataFrame
df.head(10)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

,Session,Session Topic,Title,Authors,Affiliations,Chair Name,Chair Affiliation
0,Day One,Day One,Day One,14th December,14th December (Day One,14th December,14th December (Day One
1,Introductory Remarks,Introductory Remarks,Introductory Remarks,Introductory Remarks,Introductory Remarks,Introductory Remarks,Introductory Remarks
2,December 14,December 14,December 14,December 14,December 14,December 14,December 14
3,•,•,•,•,•,•,•
4,link/room,link/room,Session link/room,Session link/room,Session link/room,Session,Session
5,Delhi School of Economics,Delhi School of Economics,"Aditya Bhattacharjea, Delhi School of Economics","Aditya Bhattacharjea, Delhi School of Economics","Aditya Bhattacharjea, Delhi School of Economics",Aditya Bhattacharjea,"Aditya Bhattacharjea, Delhi School of Economics"
6,Invited talks,Invited talks,Invited talks,Invited talks,Invited talks,Invited talks,Invited talks
7,December 14,December 14,"08:00-09:30, December 14",December 14,December 14,December 14,December 14
8,•,•,•,•,•,•,•
9,Link/room,Link/room,Session Link/room,Link/room,Session Link/room,Link,Session


In [29]:
import re
import pandas as pd

# Load the text file content
file_path = "/content/dse2020_removed.txt"
with open(file_path, 'r', encoding='utf-8') as file:
    content = file.read()

# Regular expressions for matching
session_regex = r"(Session \d+\(Room \d+\)|Session Link/Room|session|Session)"
title_regex = r"•\s*(.*)"
author_affiliation_regex = r"(.*?)(?:\((.*?)\))"  # Regex for extracting authors and affiliations
chair_regex = r"Chair:\s*(.*?)(?:\((.*?)\))?"
time_room_regex = r"\d{1,2}:\d{2}"  # Time regex (to skip time/room details)
bullet_point = "•"

# Initialize storage for results
data = []

# Split the content into lines
lines = content.splitlines()

# Track session context
current_session = None
current_topic = None
current_chair = None
current_chair_affiliation = None

# Loop through the lines and extract information
for i, line in enumerate(lines):
    line = line.strip()

    # Skip empty lines or irrelevant lines (time, room links)
    if not line or re.match(time_room_regex, line):
        continue

    # Match session (session starts with "Session", "Session x(Room y)", or similar)
    if re.match(session_regex, line, re.IGNORECASE):
        current_session = line  # Mark session start
        current_topic = None  # Reset session topic
        current_chair = None
        current_chair_affiliation = None
        continue  # Move to the next line

    # Extract session topic (line after session)
    if current_session and not current_topic:
        if line.startswith(bullet_point):  # Next bullet point after session
            current_topic = line[1:].strip()  # Session topic
        continue

    # Skip lines with time/room details
    if "room/link" in line.lower():
        continue

    # Match paper title after skipping time/room lines
    if line.startswith(bullet_point) and current_topic:
        paper_title = line[1:].strip()  # Paper title

        # Check for authors and affiliations (next line after title)
        authors = []
        affiliations = []
        if i + 1 < len(lines):
            next_line = lines[i + 1].strip()
            if not next_line.startswith(bullet_point):  # If not a bullet point, it's author info
                matches = re.findall(author_affiliation_regex, next_line)
                for author, affiliation in matches:
                    authors.append(author.strip())
                    affiliations.append(affiliation.strip())

        # Check for chairperson info (next relevant line)
        chair_found = False
        if i + 1 < len(lines):
            next_line = lines[i + 1].strip()
            if next_line.lower().startswith("chair:"):
                chair_match = re.match(chair_regex, next_line)
                if chair_match:
                    current_chair = chair_match.group(1).strip()
                    current_chair_affiliation = chair_match.group(2).strip() if chair_match.group(2) else None
                    chair_found = True

        # Append data for the current session paper
        data.append({
            "Session": current_session,
            "Session Topic": current_topic,
            "Title": paper_title,
            "Authors": ", ".join(authors),
            "Affiliations": ", ".join(affiliations),
            "Chair Name": current_chair if chair_found else "None",
            "Chair Affiliation": current_chair_affiliation if chair_found else "None"
        })

# Convert the collected data into a DataFrame
df = pd.DataFrame(data)

# Save the DataFrame to a CSV file
output_path = "/content/extracted_sessions_clean.csv"
df.to_csv(output_path, index=False)

# Display the first few rows of the DataFrame
df.head(10)


,Session,Session Topic,Title,Authors,Affiliations,Chair Name,Chair Affiliation
0,Session Link/room,Health and Development,,,,None,None
1,Session Link/room,Global Value Chains: The Economics of Spiders ...,,,,None,None
2,Session link/room,Global Value Chains: The Economics of Spiders ...,,,,None,None
3,Session 1(Room 1),Topic- Choice: Theory,Independence and Lexicographic Expected Utility,,,None,None
4,Session 1(Room 1),Topic- Choice: Theory,Choice by Rejection,,,None,None
5,Session 1(Room 1),Topic- Choice: Theory,Anticipation and Temptation,,,None,None
6,Session 1(Room 2),Topic- Industrial Organization,What Drives Efficiency in RideSharing Markets?...,,,None,None
7,Session 1(Room 2),Topic- Industrial Organization,Production Networks and Learning-by-Networking,,,None,None
8,Session 1(Room 2),Topic- Industrial Organization,Building up Trust in a Dynamic Game: A study o...,,,None,None
9,Session 1(Room 3),Topic- Global Outsourcing,The Dynamics of Global Sourcing,,,None,None


# Collect Plenary session chairs

In [ ]:
import re
import pandas as pd

# Assuming the text is stored in the variable "text"
conference_year = "2018"  # You can directly set the conference year here

# Initialize an empty list to store the extracted data
data = []

# Split the text into lines for easier processing
lines = text.split('\n')

# Initialize variables to store current session theme and chair
current_session_theme = None

# Iterate through each line to extract data based on the given cues
for line in lines:
    session_match = re.search(r'(Session\s*\d+\([a-z]\):\s*[^:]+)', line, re.IGNORECASE)
    chair_match = re.search(r'Chair:\s*(.*)', line, re.IGNORECASE)

    if session_match:
        current_session_theme = session_match.group().strip()
    elif chair_match and current_session_theme:
        chair = chair_match.group(1).strip()
        data.append([conference_year, chair, current_session_theme])
        current_session_theme = None  # Reset for next session

# Create a DataFrame from the extracted data
df = pd.DataFrame(data, columns=['Conference Year', 'Name of Chair', 'Session Theme'])

# Display the DataFrame
print(df)



Empty DataFrame
Columns: [Conference Year, Name of Chair, Session Theme]
Index: []


In [ ]:
import re
import pandas as pd

# Assuming the text is stored in the variable "text"
conference_year = "2018"  # You can directly set the conference year here

# Initialize an empty list to store the extracted data
data = []

# Use regular expressions to find all session details
pattern = re.compile(r'Session\s*\d+\([a-z]\):\s*(.*?)\s*\(.*?\)\s*Chair:\s*(.*?)\n')

# Loop through all matches to extract the relevant information
for match in re.finditer(pattern, text):
    session_theme = match.group(1).strip()
    chair = match.group(2).strip()
    data.append([conference_year, chair, session_theme])

# Create a DataFrame from the extracted data
df = pd.DataFrame(data, columns=['Conference Year', 'Name of Chair', 'Session Theme'])

# Display the DataFrame
print(df)


Empty DataFrame
Columns: [Conference Year, Name of Chair, Session Theme]
Index: []


In [ ]:
import re
import pandas as pd

# Assuming the text is stored in the variable "text"
conference_year = "2023"  # You can directly set the conference year here

# Initialize an empty list to store the extracted data
data = []

# Split the text into lines for easier processing
lines = text.split('\n')

# Initialize variables to store current session theme and chair
current_session_theme = None

# Iterate through each line to extract data based on the given cues
for i, line in enumerate(lines):
    if "Session" in line:
        current_session_theme = line.strip()
    if "Chair:" in line:
        chair = line.split("Chair:")[1].strip()
        if current_session_theme:
            data.append([conference_year, chair, current_session_theme])
            current_session_theme = None  # Reset for next session

# Create a DataFrame from the extracted data
df = pd.DataFrame(data, columns=['Conference Year', 'Name of Chair', 'Session Theme'])

# Display the DataFrame
print(df)


   Conference Year            Name of Chair  \
0             2023             Sandhya Garg   
1             2023           Michael Palmer   
2             2023            Utsa Banerjee   
3             2023        Ashwini Deshpande   
4             2023                Sankar De   
..             ...                      ...   
69            2023  Vipin Pudiyadath Veetil   
70            2023           Manish Chauhan   
71            2023              Ruchi Singh   
72            2023       Prasenjit Banerjee   
73            2023           Shanjukta Nath   

                                        Session Theme  
0   Session 1(a) Social Identity and Financial Inc...  
1   Session 1(b) Welfare Programmes I (Seminar Roo...  
2   Session 1(c) Gender I: Women's Safety (Seminar...  
3        Session 1(d) Religion and Caste (Auditorium)  
4   Session 1(e) Real Estate and Finance I (Classr...  
..                                                ...  
69  Parallel Sessions 8 (continued…)       

In [ ]:
import re
import pandas as pd

# Assuming the text is stored in the variable "text"
conference_year = "2023"  # You can directly set the conference year here

# Initialize an empty list to store the extracted data
data = []

# Split the text into lines for easier processing
lines = text.split('\n')

# Initialize variables to store current session theme and chair
current_session_theme = None

# Iterate through each line to extract data based on the given cues
for i, line in enumerate(lines):
    if "Session" in line:
        current_session_theme = line.strip()
    if "Chair:" in line:
        chair = line.split("Chair:")[1].strip()
        if current_session_theme:
            data.append([conference_year, chair, current_session_theme])
            current_session_theme = None  # Reset for next session

# Create a DataFrame from the extracted data
df = pd.DataFrame(data, columns=['Conference Year', 'Name of Chair', 'Session Theme'])

# Refine the 'Session Theme' column to remove prefixes
df['Session Theme'] = df['Session Theme'].apply(lambda x: re.sub(r'Session\s*\d+\([a-z]\):\s*', '', x).strip())

# Display the refined DataFrame
print(df)


   Conference Year            Name of Chair  \
0             2023             Sandhya Garg   
1             2023           Michael Palmer   
2             2023            Utsa Banerjee   
3             2023        Ashwini Deshpande   
4             2023                Sankar De   
..             ...                      ...   
69            2023  Vipin Pudiyadath Veetil   
70            2023           Manish Chauhan   
71            2023              Ruchi Singh   
72            2023       Prasenjit Banerjee   
73            2023           Shanjukta Nath   

                                        Session Theme  
0   Session 1(a) Social Identity and Financial Inc...  
1   Session 1(b) Welfare Programmes I (Seminar Roo...  
2   Session 1(c) Gender I: Women's Safety (Seminar...  
3        Session 1(d) Religion and Caste (Auditorium)  
4   Session 1(e) Real Estate and Finance I (Classr...  
..                                                ...  
69  Parallel Sessions 8 (continued…)       

In [ ]:
import re

# Assuming df is your existing DataFrame
def clean_session_theme(session_theme):
    # Remove the initial "Session x(x): " part
    cleaned_theme = re.sub(r'Session\s*\d+\([a-z]\):\s*', '', session_theme)
    # Remove everything in parentheses and subsequent text
    cleaned_theme = re.sub(r'\(.*?\)', '', cleaned_theme).strip()
    # Remove remaining "Session x" patterns
    cleaned_theme = re.sub(r'Session\s*\d+\s*', '', cleaned_theme).strip()
    # Remove remaining "Session x" patterns
    cleaned_theme = re.sub(r'Parallel Session\s*\d+\s*', '', cleaned_theme).strip()
    return cleaned_theme

df['Session Theme'] = df['Session Theme'].apply(clean_session_theme)

# Display the refined DataFrame
print(df)


   Conference Year            Name of Chair  \
0             2023             Sandhya Garg   
1             2023           Michael Palmer   
2             2023            Utsa Banerjee   
3             2023        Ashwini Deshpande   
4             2023                Sankar De   
..             ...                      ...   
69            2023  Vipin Pudiyadath Veetil   
70            2023           Manish Chauhan   
71            2023              Ruchi Singh   
72            2023       Prasenjit Banerjee   
73            2023           Shanjukta Nath   

                                        Session Theme  
0             Social Identity and Financial Inclusion  
1                                Welfare Programmes I  
2                            Gender I: Women's Safety  
3                                  Religion and Caste  
4                           Real Estate and Finance I  
..                                                ...  
69  Parallel Sessions 8               20 De

In [ ]:
import re

# Assuming df is your existing DataFrame
def clean_session_theme(session_theme):
    # Remove the initial "Session x(x): " part
    cleaned_theme = re.sub(r'Session\s*\d+\([a-z]\):\s*', '', session_theme)
    # Remove remaining "Session x" patterns
    cleaned_theme = re.sub(r'Session\s*\d+\s*', '', cleaned_theme).strip()
    return cleaned_theme

df['Session Theme'] = df['Session Theme'].apply(clean_session_theme)

# Display the refined DataFrame
print(df)


   Conference Year            Name of Chair  \
0             2023             Sandhya Garg   
1             2023           Michael Palmer   
2             2023            Utsa Banerjee   
3             2023        Ashwini Deshpande   
4             2023                Sankar De   
..             ...                      ...   
69            2023  Vipin Pudiyadath Veetil   
70            2023           Manish Chauhan   
71            2023              Ruchi Singh   
72            2023       Prasenjit Banerjee   
73            2023           Shanjukta Nath   

                                        Session Theme  
0             Social Identity and Financial Inclusion  
1                                Welfare Programmes I  
2                            Gender I: Women's Safety  
3                                  Religion and Caste  
4                           Real Estate and Finance I  
..                                                ...  
69  Parallel Sessions 8               20 De

Save the conference chairs in the second sheet

In [ ]:
import pandas as pd
# Install xlsxwriter if not already installed
!pip install xlsxwriter

# Load the original CSV file into a DataFrame
original_df = pd.read_csv('/content/drive/My Drive/ISI2023.csv')


# Create a Pandas Excel writer using XlsxWriter as the engine
with pd.ExcelWriter('ISI2023.xlsx', engine='xlsxwriter') as writer:
    # Write the original DataFrame to the first sheet
    original_df.to_excel(writer, sheet_name='Sheet1', index=False)
    # Write the new DataFrame to the second sheet
    df.to_excel(writer, sheet_name='Sheet2', index=False)

print("Data has been written to ISI2018.xlsx with two sheets.")


Data has been written to ISI2018.xlsx with two sheets.


Remove 'Chair' sections

In [ ]:
import re
# Clean the text by removing lines that start with 'Chair:'
text = re.sub(r"Chair:.*\n", "", text)

# Print the cleaned text
print(text)


Parallel Sessions 1                                              19 December, Monday 11:50 AM – 01:00  PM    
Session 1(a) Empirical Trade I (Conference Room) 
    Yogeshwar Bharat (Centre For Advanced Financial Research And Learning) & Raoul Minetti (Michigan State 
University): "Credit Constraints, Bank Incentives, and Firm Export: Evidence from India" 

Louise Guillouet (Organisation for Economic Co-operation and Development), Rocco Macchiavello (London 
School of Economics), Amit Kumar Khandelwal (Yale University) & Matthieu Teachout (Pôle Emploi): 
"Language Barriers in Multinationals and Knowledge Transfers" 

Session 1(b) Empirical IO (Seminar Room 1) 
Apoorva Gupta (University of Duesseldorf) & Joel Stiebale (University of Duesseldorf): "Gains from Patent 
Protection - Evidence from India" 

    
Kei Kawai (University of California Berkeley), Jun Nakabayashi (Kyoto University) & Daichi Shimamoto 
(Kindai University): "A Study of Bid-rigging in Procurement Auctions: Evidence fro

Insert Paper Markers

In [ ]:
import re

# Define the function to add 'begin paper' and 'end paper' markers
def add_paper_markers(text):
    # Add 'paper' after each paper title
    text = re.sub(r'("[^"]+")', r'\1 paper', text)

    # Add 'begin paper' before the first author
    text = re.sub(r'(\n[A-Za-z\s]+ \([^)]+\))', r'\nbegin paper \1', text)

    # Replace ' paper' with ' end paper' after the title
    text = re.sub(r'(".*?") paper', r'\1 end paper', text)

    return text

# Apply the transformation to the input text
text_with_markers = add_paper_markers(text)

# Print the transformed text with markers
print(text_with_markers)


Parallel Sessions 1                                              19 December, Monday 11:50 AM – 01:00  PM    
Session 1(a) Empirical Trade I (Conference Room) 
begin paper 
    Yogeshwar Bharat (Centre For Advanced Financial Research And Learning) & Raoul Minetti (Michigan State 
University): "Credit Constraints, Bank Incentives, and Firm Export: Evidence from India" end paper 
begin paper 

Louise Guillouet (Organisation for Economic Co-operation and Development), Rocco Macchiavello (London 
School of Economics), Amit Kumar Khandelwal (Yale University) & Matthieu Teachout (Pôle Emploi): 
"Language Barriers in Multinationals and Knowledge Transfers" end paper 

Session 1(b) Empirical IO (Seminar Room 1) 
begin paper 
Apoorva Gupta (University of Duesseldorf) & Joel Stiebale (University of Duesseldorf): "Gains from Patent 
Protection - Evidence from India" paper 
begin paper 

    
Kei Kawai (University of California Berkeley), Jun Nakabayashi (Kyoto University) & Daichi Shimamoto 
(Kin

# Scrapping ISI: 5th attempt: NLP 'en_core_web_sm'

## Process

In [ ]:
!pip install spacy pandas
import spacy
import pandas as pd

# Load the spaCy model
nlp = spacy.load("en_core_web_sm")

In [ ]:
import spacy
import pandas as pd
import re

# Load the English NLP model
nlp = spacy.load("en_core_web_sm")

# Define a function to extract session details using NLP
def extract_session_details(text):
    doc = nlp(text)

    sessions = []

    # Regular expression to identify sessions and topics
    session_re = re.compile(r"Session (\d+[a-z]*)([^\n]*)\s*\(([^)]*)\)")

    # Regular expression to match the authors and affiliations
    author_re = re.compile(r"([A-Za-z\s]+)\s?\(([^)]+)\)")

    # Regular expression to capture paper title
    title_re = re.compile(r"\"([^\"]+)\"")

    # Iterate over all sessions in the text
    for match in session_re.finditer(text):
        session_number, session_topic, location = match.groups()

        # Locate the paper(s) associated with the session
        session_start = match.end()
        session_end = text.find("Session", session_start)  # Find next session to stop the current one
        session_content = text[session_start:session_end] if session_end != -1 else text[session_start:]

        # Extract the paper titles
        paper_titles = title_re.findall(session_content)

        # Extract authors and their affiliations
        authors_and_affiliations = author_re.findall(session_content)

        # Process each paper
        for paper_title in paper_titles:
            authors = []
            affiliations = []

            for author_name, affiliation in authors_and_affiliations:
                authors.append(author_name.strip())
                affiliations.append(affiliation.strip())

            # Loop over authors to create a row for each
            for author, affiliation in zip(authors, affiliations):
                sessions.append({
                    "Session": session_number,
                    "Topic": session_topic,
                    "Title of Paper": paper_title,
                    "Number of Authors": len(authors),
                    "Author": author,
                    "Institute/Affiliation": affiliation
                })

    # Convert the session details into a DataFrame
    df = pd.DataFrame(sessions)

    return df


# Extract session details from the conference text
conference_df2 = extract_session_details(text)

# Display the structured DataFrame
print(conference_df2)

# Save to CSV
conference_df2.to_csv("/content/drive/My Drive/conference_details_draft.csv", index=False)


     Session                   Topic  \
0          1  (a) Empirical Trade I    
1          1  (a) Empirical Trade I    
2          1  (a) Empirical Trade I    
3          1  (a) Empirical Trade I    
4          1  (a) Empirical Trade I    
...      ...                     ...   
1470       8        (i) Finance III    
1471       8        (i) Finance III    
1472       8        (i) Finance III    
1473       8        (i) Finance III    
1474       8        (i) Finance III    

                                         Title of Paper  Number of Authors  \
0     Credit Constraints, Bank Incentives, and Firm ...                  6   
1     Credit Constraints, Bank Incentives, and Firm ...                  6   
2     Credit Constraints, Bank Incentives, and Firm ...                  6   
3     Credit Constraints, Bank Incentives, and Firm ...                  6   
4     Credit Constraints, Bank Incentives, and Firm ...                  6   
...                                                

In [ ]:
import spacy
import pandas as pd
import re

# Load the English NLP model
nlp = spacy.load("en_core_web_sm")

# Define a function to extract session details using NLP
def extract_session_details(text):
    doc = nlp(text)

    sessions = []

    # Regular expression to identify sessions and topics
    session_re = re.compile(r"Session (\d+[a-z]*)([^\n]*)\s*\(([^)]*)\)")

    # Regular expression to match the authors and affiliations
    author_re = re.compile(r"([A-Za-z\s]+)\s?\(([^)]+)\)")

    # Regular expression to capture paper title with 'begin paper' and 'end paper'
    paper_re = re.compile(r'begin paper(.*?)paper', re.DOTALL)

    # Iterate over all sessions in the text
    for match in session_re.finditer(text):
        session_number, session_topic, location = match.groups()

        # Locate the paper(s) associated with the session
        session_start = match.end()
        session_end = text.find("Session", session_start)  # Find next session to stop the current one
        session_content = text[session_start:session_end] if session_end != -1 else text[session_start:]

        # Extract the paper sections
        paper_sections = paper_re.findall(session_content)

        # Process each paper section
        for paper_section in paper_sections:
            authors = []
            affiliations = []

            # Extract the paper title
            title_match = re.search(r'\"([^\"]+)\"', paper_section)
            if title_match:
                paper_title = title_match.group(1)

            # Extract authors and their affiliations for the current paper
            for match in author_re.finditer(paper_section):
                author_name = match.group(1).strip()
                affiliation = match.group(2).strip()
                authors.append(author_name)
                affiliations.append(affiliation)

            # Loop over authors to create a row for each
            for author, affiliation in zip(authors, affiliations):
                sessions.append({
                    "Session": session_number,
                    "Topic": session_topic,
                    "Title of Paper": paper_title,
                    "Number of Authors": len(authors),
                    "Author": author,
                    "Institute/Affiliation": affiliation
                })

    # Convert the session details into a DataFrame
    df = pd.DataFrame(sessions)

    return df

# Apply the transformation to the input text
text_with_markers = add_paper_markers(text)

# Extract session details from the transformed conference text
conference_df = extract_session_details(text_with_markers)

# Display the structured DataFrame
print(conference_df)

# Save to CSV
conference_df.to_csv("/content/drive/My Drive/conference_details.csv", index=False)

    Session                   Topic  \
0         1  (a) Empirical Trade I    
1         1  (a) Empirical Trade I    
2         1  (a) Empirical Trade I    
3         1  (a) Empirical Trade I    
4         1  (a) Empirical Trade I    
..      ...                     ...   
450       8        (i) Finance III    
451       8        (i) Finance III    
452       8        (i) Finance III    
453       8        (i) Finance III    
454       8        (i) Finance III    

                                        Title of Paper  Number of Authors  \
0    Credit Constraints, Bank Incentives, and Firm ...                  2   
1    Credit Constraints, Bank Incentives, and Firm ...                  2   
2    Language Barriers in Multinationals and Knowle...                  4   
3    Language Barriers in Multinationals and Knowle...                  4   
4    Language Barriers in Multinationals and Knowle...                  4   
..                                                 ...               

Remove unwanted characters and separate authors

In [ ]:

import pandas as pd

# Load the conference details CSV
df = pd.read_csv("/content/drive/My Drive/conference_details.csv")

# Force 'Author' column to be in string format
df['Author'] = df['Author'].astype(str)

# Function to process the 'Author' column
def process_authors(df):
    expanded_rows = []
    for index, row in df.iterrows():
        authors = row['Author']

        # Remove 'and' at the beginning
        if authors.startswith('and'):
            authors = authors[3:].strip()

        # Split authors by 'and' and '\n' where appropriate
        split_authors = re.split(r'\band\b|\n', authors)
        split_authors = [author.strip() for author in split_authors if author.strip()]

        # Create a new row for each author
        for author in split_authors:
            new_row = row.copy()
            new_row['Author'] = author
            expanded_rows.append(new_row)

    # Create a new DataFrame with the expanded rows
    expanded_df = pd.DataFrame(expanded_rows)

    return expanded_df

# Process the authors and expand the DataFrame
expanded_df = process_authors(df)

# Display the expanded DataFrame
print(expanded_df)

# Save the expanded DataFrame to a new CSV
expanded_df.to_csv("/content/drive/My Drive/expanded_conference_details.csv", index=False)


     Session                   Topic  \
0          1  (a) Empirical Trade I    
1          1  (a) Empirical Trade I    
2          1  (a) Empirical Trade I    
3          1  (a) Empirical Trade I    
4          1  (a) Empirical Trade I    
..       ...                     ...   
450        8        (i) Finance III    
451        8        (i) Finance III    
452        8        (i) Finance III    
453        8        (i) Finance III    
454        8        (i) Finance III    

                                        Title of Paper  Number of Authors  \
0    Credit Constraints, Bank Incentives, and Firm ...                  2   
1    Credit Constraints, Bank Incentives, and Firm ...                  2   
2    Language Barriers in Multinationals and Knowle...                  4   
3    Language Barriers in Multinationals and Knowle...                  4   
4    Language Barriers in Multinationals and Knowle...                  4   
..                                                 ...   

Create unique authors and remove duplicates per Paper Title

In [ ]:
import pandas as pd

# Load the updated conference details CSV
df = pd.read_csv("/content/drive/My Drive/expanded_conference_details.csv")

# Function to clean the 'Title of Paper' column by removing new lines and merging with a space
def clean_titles(df):
    df['Title of Paper'] = df['Title of Paper'].str.replace('\n', ' ')
    return df

# Clean the 'Title of Paper' column
cleaned_df = clean_titles(df)

# Display the cleaned DataFrame to verify the changes
print(cleaned_df)


     Session                   Topic  \
0          1  (a) Empirical Trade I    
1          1  (a) Empirical Trade I    
2          1  (a) Empirical Trade I    
3          1  (a) Empirical Trade I    
4          1  (a) Empirical Trade I    
..       ...                     ...   
452        8        (i) Finance III    
453        8        (i) Finance III    
454        8        (i) Finance III    
455        8        (i) Finance III    
456        8        (i) Finance III    

                                        Title of Paper  Number of Authors  \
0    Credit Constraints, Bank Incentives, and Firm ...                  2   
1    Credit Constraints, Bank Incentives, and Firm ...                  2   
2    Language Barriers in Multinationals and Knowle...                  4   
3    Language Barriers in Multinationals and Knowle...                  4   
4    Language Barriers in Multinationals and Knowle...                  4   
..                                                 ...   

In [ ]:
print(cleaned_df)

     Session                   Topic  \
0          1  (a) Empirical Trade I    
1          1  (a) Empirical Trade I    
2          1  (a) Empirical Trade I    
3          1  (a) Empirical Trade I    
4          1  (a) Empirical Trade I    
..       ...                     ...   
452        8        (i) Finance III    
453        8        (i) Finance III    
454        8        (i) Finance III    
455        8        (i) Finance III    
456        8        (i) Finance III    

                                        Title of Paper  Number of Authors  \
0    Credit Constraints, Bank Incentives, and Firm ...                  2   
1    Credit Constraints, Bank Incentives, and Firm ...                  2   
2    Language Barriers in Multinationals and Knowle...                  4   
3    Language Barriers in Multinationals and Knowle...                  4   
4    Language Barriers in Multinationals and Knowle...                  4   
..                                                 ...   

## Save dataset

In [ ]:
# Save the cleaned and updated DataFrame to a new CSV
cleaned_df.to_csv("/content/drive/My Drive/ISI2022.csv", index=False)

# Adding Countries

In [ ]:
!pip install pandas geopy openpyxl tqdm


In [ ]:
!ls /content/drive/MyDrive/ISI/

2018_removed.pdf  2021_removed.pdf  2023_removed.pdf  ISI2019.xlsx  ISI2022.xlsx  ISIclean
2019_removed.pdf  2022_removed.pdf  ISI2018.xlsx      ISI2021.xlsx  ISI2023.xlsx


In [ ]:
# Load Excel file
file_path = '/content/drive/MyDrive/ISI/ISI2023.xlsx'  # Replace with the path to your file
xls = pd.ExcelFile(file_path)

In [ ]:
import pandas as pd
# List all sheets and their columns
for sheet_name in xls.sheet_names:
    df = pd.read_excel(xls, sheet_name=sheet_name)
    print(f"Sheet: {sheet_name}")
    print(f"Columns: {df.columns.tolist()}")
    print("-" * 40)  # Separator for better readability


Sheet: Authrowise (2023)
Columns: ['Session', 'Topic', 'Title of Paper', 'Number of Authors', 'Author', 'Institute/Affiliation']
----------------------------------------
Sheet: Chair of regular sessions
Columns: ['Conference Year', 'Name of Chair', 'Session Theme', 'Institute/Affiliation']
----------------------------------------
Sheet: SPECIAL sessions Keynote, Plena
Columns: ['Conference Year', 'Session Type', 'Session Number', 'Presenter', 'Topic', 'Affiliation']
----------------------------------------


In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm
import openpyxl

# Initialize geolocator with a rate limiter
geolocator = Nominatim(user_agent="geo_affiliation_locator")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

def get_country_from_affiliation(affiliation):
    """
    Get the country name from the affiliation string using geopy, ensuring results are in English.
    """
    try:
        if pd.isnull(affiliation) or affiliation.strip() == "":
            return None
        location = geocode(affiliation, timeout=10)
        if location and location.raw.get('display_name'):
            address_parts = location.raw['display_name'].split(",")
            # Extract and clean the country name
            country = address_parts[-1].strip()
            return country
        return None
    except Exception as e:
        return None

# Get all sheets
all_sheets = {sheet_name: pd.read_excel(xls, sheet_name=sheet_name) for sheet_name in xls.sheet_names}

# Sheets to process
sheets_to_process = ["Authrowise (2023)", "Chair of regular sessions", "SPECIAL sessions Keynote, Plena"]

# Process each specified sheet
for sheet in sheets_to_process:
    print(f"Processing sheet: {sheet}")
    df = all_sheets[sheet]

    # Identify the relevant columns
    affiliation_col = "Affiliation" if "Affiliation" in df.columns else "Institute/Affiliation"
    country_col = "Affiliation Country"

    # Add the country column if it doesn't exist
    if country_col not in df.columns:
        df[country_col] = None

    # Fill the 'Affiliation Country' column
    tqdm.pandas(desc=f"Processing {sheet}")
    df[country_col] = df[affiliation_col].progress_apply(get_country_from_affiliation)

    # Update the sheet in the dictionary
    all_sheets[sheet] = df

# Save back to Excel, preserving column widths
output_path = "ISI_updated1.xlsx"
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, sheet_df in all_sheets.items():
        sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)
        # Retain column widths
        workbook = writer.book
        worksheet = writer.sheets[sheet_name]
        for col_num, column_cells in enumerate(sheet_df.columns, start=1):
            max_length = max(sheet_df[column_cells].astype(str).map(len).max(), len(column_cells))
            worksheet.column_dimensions[openpyxl.utils.get_column_letter(col_num)].width = max_length

print(f"Processing complete. Updated file saved as {output_path}.")


Processing sheet: Authrowise (2023)


Processing Authrowise (2023):  65%|██████▍   | 323/500 [05:26<02:57,  1.00s/it]WARNING:geopy:RateLimiter caught an error, retrying (0/2 tries). Called with (*('Beijing University',), **{'timeout': 10}).
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/geopy/geocoders/base.py", line 368, in _call_geocoder
    result = self.adapter.get_json(url, timeout=timeout, headers=req_headers)
  File "/usr/local/lib/python3.10/dist-packages/geopy/adapters.py", line 472, in get_json
    resp = self._request(url, timeout=timeout, headers=headers)
  File "/usr/local/lib/python3.10/dist-packages/geopy/adapters.py", line 500, in _request
    raise AdapterHTTPError(
geopy.adapters.AdapterHTTPError: Non-successful status code 503

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/geopy/extra/rate_limiter.py", line 136, in _retries_gen
    yield i  # Run the function.
  F

Processing sheet: Chair of regular sessions


Processing Chair of regular sessions: 100%|██████████| 74/74 [01:16<00:00,  1.03s/it]


Processing sheet: SPECIAL sessions Keynote, Plena


Processing SPECIAL sessions Keynote, Plena: 100%|██████████| 6/6 [00:06<00:00,  1.16s/it]

Processing complete. Updated file saved as ISI_updated1.xlsx.


Determine location in India or not

In [ ]:
import pandas as pd
import openpyxl

# Load Excel file
file_path = "/content/drive/MyDrive/ISI/ISIclean/ISI2022.xlsx"  # Replace with the path to your file
xls = pd.ExcelFile(file_path)

# Get all sheets
all_sheets = {sheet_name: pd.read_excel(xls, sheet_name=sheet_name) for sheet_name in xls.sheet_names}

# Process all sheets
for sheet in all_sheets.keys():
    print(f"Processing sheet: {sheet}")
    df = all_sheets[sheet]

    # Check if the 'Affiliation Country' column exists
    if "Affiliation Country" not in df.columns:
        print(f"Error: 'Affiliation Country' column not found in {sheet}. Skipping.")
        continue

    # Create the new column 'Affiliation Location'
    df['Affiliation Location'] = df['Affiliation Country'].apply(lambda x: 1 if x == "India" else 0)

    # Update the sheet in the dictionary
    all_sheets[sheet] = df

# Save back to Excel, preserving column widths
output_path = "/content/drive/MyDrive/ISI/ISIclean/ISI2019.xlsx"
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, sheet_df in all_sheets.items():
        sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)
        # Retain column widths
        workbook = writer.book
        worksheet = writer.sheets[sheet_name]
        for col_num, column_cells in enumerate(sheet_df.columns, start=1):
            max_length = max(sheet_df[column_cells].astype(str).map(len).max(), len(column_cells))
            worksheet.column_dimensions[openpyxl.utils.get_column_letter(col_num)].width = max_length

print(f"Processing complete. Updated file with 'Affiliation Location' column saved as {output_path}.")


Processing sheet: Authorwise (2019)
Processing sheet: Chair of regular sessions
Processing sheet: SPECIAL sessions Keynote, Plena
Processing complete. Updated file with 'Affiliation Location' column saved as /content/drive/MyDrive/ISI/ISIclean/ISI2019.xlsx.


Add genders

In [ ]:
!pip install requests
!pip install gender-guesser
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 379.3/379.3 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 10.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import gender_guesser.detector as gender
from tqdm import tqdm

# Function to get gender using gender-guesser
def get_gender(name):
    if not isinstance(name, str):  # Check if the name is a string
        return 'unknown'

    first_name = name.split()[0]  # Get first name
    d = gender.Detector()
    gender_guess = d.get_gender(first_name)

    # Convert the gender guess to a more readable format
    if gender_guess in ['male', 'mostly_male']:
        return 'Male'
    elif gender_guess in ['female', 'mostly_female']:
        return 'Female'
    else:
        return 'unknown'

# Load the Excel file
file_path = "/content/drive/MyDrive/ISI/ISIclean/ISI2019.xlsx"  # Replace with your actual file path
xls = pd.ExcelFile(file_path)

# List of sheets and the corresponding column names to process
sheet_columns = {
    'Authorwise (2019)': 'Author',
    'SPECIAL sessions Keynote, Plena': 'Name of chair',
    'Chair of regular sessions': 'Presenter/Panelist/Moderator'
}

# Iterate through each sheet and process it
with pd.ExcelWriter("processed_ISI.xlsx", engine='openpyxl') as writer:
    for sheet_name in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name=sheet_name)

        # Check if the column exists in the current sheet
        if sheet_name in sheet_columns and sheet_columns[sheet_name] in df.columns:
            column_name = sheet_columns[sheet_name]

            # Use tqdm for a progress bar while processing rows
            tqdm.pandas(desc=f"Processing {sheet_name}")

            # Apply gender detection for the specified column
            df['Gender'] = df[column_name].progress_apply(get_gender)

        # Write the DataFrame back to the Excel writer
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print("Processing completed!")


Processing Authorwise (2019): 100%|██████████| 369/369 [02:11<00:00,  2.81it/s]

Processing completed!


# Experimental

In [ ]:
import spacy
import re

# Load the English NLP model
nlp = spacy.load("en_core_web_sm")

# Function to process text with NLP and add markers
def add_markers_with_nlp(text):
    doc = nlp(text)
    updated_text = []
    delimiter = "***NEW PAPER***"

    session_re = re.compile(r'(Session \d+[a-z]* [^\n]* \([^\)]*\)\n)')
    session_blocks = session_re.split(text)

    for i, block in enumerate(session_blocks):
        if i % 2 == 0:
            updated_text.append(block)
        else:
            papers = re.split(r'(?:"[^"]+"\s*\n)', block)  # Split block at paper titles
            for j, paper in enumerate(papers):
                if j == 0:
                    updated_text.append(paper)
                else:
                    updated_text.append(f'{delimiter} "{papers[j-1].split(": ")[-1].strip()}" {paper}')

    return ''.join(updated_text)

# Assuming 'text' is already loaded
updated_text = add_markers_with_nlp(text)

# Print the updated text
print(updated_text)


Parallel Sessions 1                                                18 December, Monday 11:50 AM- 01:00  PM    
Session 1(a) Social Identity and Financial Inclusion (Conference Room) 
Rikhia Bhukta (Indian Institute of Technology Kanpur), Debayan Pakrashi (Indian Statistical Institute Kolkata; 
Indian Institute of Technology Kanpur), Sarani Saha (Indian Institute of Technology Kanpur), and Ashish Sedai 
(University of Texas at Arlington): "Does Financial Inclusion Mitigate Social Exclusion? Causal Evidence from 
India" 

Sandhya Garg (Institute of Economic Growth), Samarth Gupta (Ahmedabad University), and Sushanta Mallick 
(Queen Mary University of London): "Does Social Identity Constrain Rural Entrepreneurship? The Role of 
Financial Inclusion" 
Session 1(b) Welfare Programmes I (Seminar Room 1) 
Vidhya Unnikrishnan (University of Manchester): "Impact of Social Protection Policy Targeted on the 
Disabled: Panel Data Evidence from India" 


Michael Palmer (University of Western Austral

In [ ]:
import spacy
import pandas as pd
import re

# Load the English NLP model
nlp = spacy.load("en_core_web_sm")

# Define a function to extract session details using NLP
def extract_session_details(text):
    doc = nlp(text)

    sessions = []

    # Regular expression to identify sessions and topics
    session_re = re.compile(r"Session (\d+[a-z]*)([^\n]*)\s*\(([^)]*)\)")

    # Regular expression to match the authors and affiliations
    author_re = re.compile(r"([A-Za-z\s]+)\s?\(([^)]+)\)")

    # Regular expression to capture paper titles
    title_re = re.compile(r'"\*\*\*NEW PAPER\*\*\* ([^\"]+)"')

    # Iterate over all sessions in the text
    for match in session_re.finditer(text):
        session_number, session_topic, location = match.groups()

        # Locate the paper(s) associated with the session
        session_start = match.end()
        session_end = text.find("Session", session_start)  # Find next session to stop the current one
        session_content = text[session_start:session_end] if session_end != -1 else text[session_start:]

        # Extract the paper titles
        paper_titles = title_re.findall(session_content)

        # Extract authors and their affiliations
        authors_and_affiliations = author_re.findall(session_content)

        # Process each paper
        for paper_title in paper_titles:
            authors = []
            affiliations = []

            for author_name, affiliation in authors_and_affiliations:
                authors.append(author_name.strip())
                affiliations.append(affiliation.strip())

            # Loop over authors to create a row for each
            for author, affiliation in zip(authors, affiliations):
                sessions.append({
                    "Session": session_number,
                    "Topic": session_topic,
                    "Title of Paper": paper_title,
                    "Number of Authors": len(authors),
                    "Author": author,
                    "Institute/Affiliation": affiliation
                })

    # Convert the session details into a DataFrame
    df = pd.DataFrame(sessions)

    return df

# Extract session details from the conference text
conference_df = extract_session_details(updated_text)

# Display the structured DataFrame
print(conference_df)

# Save to CSV
conference_df.to_csv("/content/drive/My Drive/conference_details.csv", index=False)


Empty DataFrame
Columns: []
Index: []
